# How to create enforcable structured outputs with LLMs

In [26]:
from dotenv import load_dotenv

load_dotenv("../../.env")

import openai
from pydantic import BaseModel, Field
import instructor
import json

In [3]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS #use responses rather than completions API
)

In [5]:
prompt = """You are a helpful assistant. 
Please answer the following question: 
What is your name?"""

In [6]:
# data classes for structure enforcement
class Answer(BaseModel):
    answer: str = Field(description="Answer to the question asked")

In [7]:
response = client.create(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [8]:
response

Answer(answer='I’m ChatGPT.')

In [9]:
response.answer

'I’m ChatGPT.'

### Create a structured response with more metadata

In [30]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [31]:
dict(raw_response)

{'id': 'resp_0b84466d26a71260006a42cd114604819d8f4cd2012a85de8e',
 'created_at': 1782762769.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-5.4-nano-2026-03-17',
 'object': 'response',
 'output': [ResponseFunctionToolCall(arguments='{"answer":"I\'m ChatGPT."}', call_id='call_AJRSveQrxo2r6kFPDNK7o3nb', name='Answer', type='function_call', id='fc_0b84466d26a71260006a42cd11f750819d8b4940a928167384', namespace=None, status='completed')],
 'parallel_tool_calls': True,
 'temperature': 1.0,
 'tool_choice': ToolChoiceFunction(name='Answer', type='function'),
 'tools': [FunctionTool(name='Answer', parameters={'properties': {'answer': {'description': 'Answer to the question asked', 'title': 'Answer', 'type': 'string'}}, 'required': ['answer'], 'title': 'Answer', 'type': 'object', 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Correctly extracted `Answer` with all the required parameters with co

In [34]:
# pure answer without metadata
json.loads(raw_response.output[0].arguments).get("answer")

"I'm ChatGPT."

In [35]:
# or
response.answer

"I'm ChatGPT."

### Use RAG with structured output

In [36]:
from qdrant_client import QdrantClient

#### Embedding function

In [37]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

#### Retrieval function

In [38]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [39]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [40]:
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [41]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

In [42]:
def generate_answer(prompt):
    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=Answer
    )

    return response

In [43]:
def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_obj": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }
    return final_answer
    

In [44]:
output = rag_pipeline("I need a powerbank")

In [45]:
output

{'data_obj': Answer(answer='The available products don’t include a power bank. They do include a rechargeable rechargeable USB mini desk fan, a rechargeable cordless air duster (USB charging), a USB C to USB C cable, and an iPhone Lightning cable, plus a foldable cruise power strip with USB C outlets. If you tell me what device you want to charge (phone/tablet/laptop) and how many times you need a full charge, I can suggest the closest option.'),
 'answer': 'The available products don’t include a power bank. They do include a rechargeable rechargeable USB mini desk fan, a rechargeable cordless air duster (USB charging), a USB C to USB C cable, and an iPhone Lightning cable, plus a foldable cruise power strip with USB C outlets. If you tell me what device you want to charge (phone/tablet/laptop) and how many times you need a full charge, I can suggest the closest option.',
 'question': 'I need a powerbank',
 'retrieved_context_ids': ['B0BM9THPDQ',
  'B0C9QZS95R',
  'B0BBVJJRHD',
  'B0BX